# Automate Event Planning

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Using python3.11.3

!pip3 install -r ../requirements.txt

/Users/namitjain/.zshenv:.:1: no such file or directory: /Users/namitjain/.cargo/env


In [3]:
from crewai import Crew, Task, Agent

In [4]:
import os
from utils import get_openai_api_key,get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

In [5]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

# Initialize the tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

In [6]:
# Agent 1: Venue Coordinator
venue_coordinator = Agent(
    role="Venue Coordinator",
    goal="Identify and book an appropriate venue "
    "based on event requirements",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "With a keen sense of space and "
        "understanding of event logistics, "
        "you excel at finding and securing "
        "the perfect venue that fits the event's theme, "
        "size, and budget constraints."
    )
)

In [7]:
logistics_manager = Agent(
    role='Logistics Manager',
    goal=(
        "Manage all logistics for the event "
        "including catering and equipment"
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Organized and detail-oriented, "
        "you ensure that every logistical aspect of the event "
        "from catering to equipment setup "
        "is flawlessly executed to create a seamless experience."
    )
)

In [8]:
marketing_communications_agent = Agent(
    role="Marketing and Communications Agent",
    goal="Effectively market the event and "
         "communicate with participants",
    tools=[search_tool, scrape_tool],
    verbose=True,
    backstory=(
        "Creative and communicative, "
        "you craft compelling messages and "
        "engage with potential attendees "
        "to maximize event exposure and participation."
    )
)

In [9]:
from pydantic import BaseModel
# Define a Pydantic model for venue details 
# (demonstrating Output as Pydantic)
class VenueDetails(BaseModel):
    name: str
    address: str
    capacity: int
    booking_status: str

## Creating Tasks
- By using `output_json`, you can specify the structure of the output you want.
- By using `output_file`, you can get your output in a file.
- By setting `human_input=True`, the task will ask for human feedback (whether you like the results or not) before finalising it.

In [11]:
venue_task = Task(
    description="Find a venue in {event_city} "
                "that meets criteria for {event_topic}.",
    expected_output="All the details of a specifically chosen"
                    "venue you found to accommodate the event.",
    human_input=True,
    output_json=VenueDetails,
    output_file="venue_details.json",  
      # Outputs the venue details as a JSON file
    agent=venue_coordinator
)

- By setting async_execution=True, it means the task can run in parallel with the tasks which come after it.

In [22]:
logistics_task = Task(
    description="Coordinate catering and "
                 "equipment for an event "
                 "with {expected_participants} participants "
                 "on {tentative_date}.",
    expected_output="Confirmation of all logistics arrangements "
                    "including catering and equipment setup.",
    human_input=True,
    async_execution=True,
    agent=logistics_manager
)

In [23]:
marketing_task = Task(
    description="Promote the {event_topic} "
                "aiming to engage at least"
                "{expected_participants} potential attendees.",
    expected_output="Report on marketing activities "
                    "and attendee engagement formatted as markdown.",
    output_file="marketing_report.md",  # Outputs the report as a text file
    agent=marketing_communications_agent
)

In [24]:
# Define the crew with agents and tasks
event_management_crew = Crew(
    agents=[venue_coordinator, 
            logistics_manager, 
            marketing_communications_agent],
    
    tasks=[venue_task, 
           logistics_task, 
           marketing_task],
    
    verbose=True
)

2025-02-10 09:59:00,061 - 8637429824 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


In [25]:
event_details = {
    'event_topic': "Tech Innovation Conference",
    'event_description': "A gathering of tech innovators "
                         "and industry leaders "
                         "to explore future technologies.",
    'event_city': "New Delhi",
    'tentative_date': "2025-02-21",
    'expected_participants': 500,
    'budget': 20000,
    'venue_type': "Conference Hall"
}

In [26]:
result = event_management_crew.kickoff(inputs=event_details)

# Agent: Venue Coordinator
## Task: Find a venue in New Delhi that meets criteria for Tech Innovation Conference.


# Agent: Venue Coordinator
## Thought: Thought: To find a suitable venue for the Tech Innovation Conference in New Delhi, I should first search for popular venue options that can accommodate tech conferences. This would involve gathering information about the venues regarding their location, capacity, and availability.
## Using tool: Search the internet with Serper
## Tool Input: 
"{\"search_query\": \"conference venues in New Delhi for tech events\"}"
## Tool Output: 
{'searchParameters': {'q': 'conference venues in New Delhi for tech events', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'IT & Technology Events in New Delhi - India - 10Times', 'link': 'https://10times.com/newdelhi-in/technology', 'snippet': 'IECC Convention Centre, New Delhi. The India Innovation Summit: Pioneering ... The Suncourt Hotel Yatri, New Delhi. ConferenceIT & Technol

In [27]:
import json
from pprint import pprint

with open('venue_details.json') as f:
   data = json.load(f)

pprint(data)

{'address': 'Sec 25, Dwarka, New Delhi 110078',
 'booking_status': 'Available',
 'capacity': 240000,
 'name': 'Yashobhoomi, India International Convention & Expo Centre (IICC)'}


In [28]:
from IPython.display import Markdown
Markdown("marketing_report.md")

# Tech Innovation Conference Marketing and Engagement Report

## Event Overview
The Tech Innovation Conference is scheduled for February 21, 2025, and aims to attract at least 500 attendees. With Swadley's Fine Catering and Marianne's Rentals providing comprehensive services, we're poised for a successful event.

## Marketing Strategies

### 1. **Event Website or Landing Page**
- **Objective:** Create an informative and engaging website/landing page for registrations.
- **Content:** Include event date, location, agenda, speakers’ bios, unique value propositions, and a clear call-to-action (CTA).
- **Tools:** Use platforms like Carrd or Swipe Pages to design the page.

### 2. **Social Media Promotion**
- **Platforms:** Leverage Instagram, LinkedIn, Facebook, and Twitter.
- **Actions:**
  - Use relevant hashtags and tag guest speakers.
  - Conduct contests or quizzes to increase engagement.
  - Share video snippets and behind-the-scenes content.

### 3. **Email Marketing Campaign**
- **Strategy:**
  - Draft personalized invitation emails with event details.
  - Send out reminders and follow-ups to increase attendance.
  - Automate emails using platforms like Mailchimp.

### 4. **Influencer Partnership**
- **Approach:**
  - Identify and partner with industry influencers to expand reach.
  - Host Instagram takeovers or live sessions with influencers.

### 5. **Advertising Campaigns**
- **Channels:** Utilize Google Ads, YouTube Ads, and Facebook for targeted marketing.
- **Focus:** Remarketing to users who interacted with initial ads.

### 6. **Press Release**
- **Distribution:** Write a compelling press release highlighting the event’s significance and distribute it to media outlets and industry blogs.

## Attendee Engagement Strategies

### 1. **Interactive Event Features**
- **Tools:** Use engagement features like polls, Q&As, and live chats during sessions to foster interaction.
- **Platforms:** Implement features on platforms capable of handling live interactions such as Livestorm.

### 2. **Networking Opportunities**
- **Action:** Organize breakout sessions and networking lounges to encourage attendee interaction.

### 3. **Survey and Feedback**
- **Post-Event Action:** Send follow-up surveys to gather attendee feedback to improve future events.

### 4. **Content Sharing**
- **Strategy:** Record and distribute event highlights or full sessions post-event for continued engagement and marketing.

## Timeline and Execution
- **Pre-Event:** Focus on building the website and launching ad campaigns.
- **During Event:** Ensure smooth live streaming and in-person experiences.
- **Post-Event:** Follow up with recording distribution and feedback collection.

## Conclusion
By employing a combination of digital marketing tactics, interactive features, and strategic partnerships, we aim to not only meet but exceed our attendee target and create a substantial impact with the Tech Innovation Conference.
```